In [ ]:
# @title 1 — Setup, Drive e caminhos — VODCA CORRECTED

%pip -q install xarray netcdf4 rasterio geopandas shapely pandas tqdm

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, date
import re, gc, json, time, warnings

import numpy as np
import pandas as pd
import xarray as xr
import rasterio as rio
import geopandas as gpd

from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================================================
# Projeto
# =========================================================
ROOT = Path("/content/drive/MyDrive/Pantanal_TippingPoints")
BASE_DIR = ROOT / "index"

# NetCDFs VODCA ORIGINAIS
RAW_VOD_DIR = Path(
    "/content/drive/MyDrive/VODCA_CXKu/"
    "VODCA_CXKu/VODCA_CXKu/"
    "daily_images_VODCA_CXKu"
)

# Limite do Pantanal
BOUNDARY_FP = ROOT / "Pantanal.shp"

# =========================================================
# NOVAS pastas — NÃO sobrescrevem as antigas
# =========================================================
VOD_BI_DIR = BASE_DIR / "interim" / "vodca_corrected"

VOD_300_DIR = (
    BASE_DIR / "outputs" /
    "vod_bi_300m_aligned_corrected"
)

LOG_DIR = BASE_DIR / "logs" / "vodca_corrected"

for d in [VOD_BI_DIR, VOD_300_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =========================================================
# Grade de referência 300 m
# IMPORTANTE:
# usar exatamente a grade óptica efetivamente usada no estudo
# =========================================================
REF_CANDIDATES = [
    Path("/content/drive/MyDrive/Pantanal_TippingPoints_optical/opt_198509.tif"),
    Path("/content/drive/My Drive/Pantanal_TippingPoints_optical/opt_198509.tif"),
]

REF_FP = next((p for p in REF_CANDIDATES if p.exists()), None)

assert REF_FP is not None, (
    "Não encontrei o raster óptico de referência opt_198509.tif."
)

# =========================================================
# Verificações
# =========================================================
assert RAW_VOD_DIR.exists(), f"VODCA original não encontrado: {RAW_VOD_DIR}"
assert BOUNDARY_FP.exists(), f"Pantanal.shp não encontrado: {BOUNDARY_FP}"

with rio.open(REF_FP) as ref:
    print("Grade 300 m de referência:")
    print(" arquivo :", REF_FP)
    print(" CRS     :", ref.crs)
    print(" EPSG    :", ref.crs.to_epsg())
    print(" shape   :", (ref.height, ref.width))
    print(" res     :", ref.res)

print("\nSaída VOD temporal corrigido:")
print(VOD_BI_DIR)

print("\nSaída VOD alinhado 300 m:")
print(VOD_300_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.8 MB/s eta 0:00:00
Mounted at /content/drive
Grade 300 m de referência:
 arquivo : /content/drive/MyDrive/Pantanal_TippingPoints_optical/opt_198509.tif
 CRS     : EPSG:31983
 EPSG    : 31983
 shape   : (2448, 1612)
 res     : (300.0, 300.0)

Saída VOD temporal corrigido:
/content/drive/MyDrive/Pantanal_TippingPoints/index/interim/vodca_corrected

Saída VOD alinhado 300 m:
/content/drive/MyDrive/Pantanal_TippingPoints/index/outputs/vod_bi_300m_aligned_corrected


In [ ]:
# @title 2 — Indexar NetCDFs originais + conferir variável e metadados

DATE_RE = re.compile(r"(19|20)\d{2}-\d{2}-\d{2}")

nc_files = sorted(RAW_VOD_DIR.rglob("*.nc"))

print(f"NetCDFs encontrados: {len(nc_files):,}")

assert len(nc_files) > 0, "Nenhum NetCDF encontrado."

# =========================================================
# Índice data -> arquivo
# =========================================================
DATE_TO_FILE = {}
duplicates = []

for fp in nc_files:
    m = DATE_RE.search(fp.name)

    if not m:
        continue

    d = pd.Timestamp(m.group(0))

    if d in DATE_TO_FILE:
        duplicates.append(d)
    else:
        DATE_TO_FILE[d] = fp

DATE_TO_FILE = dict(sorted(DATE_TO_FILE.items()))

print(f"Datas indexadas: {len(DATE_TO_FILE):,}")
print("Primeira:", next(iter(DATE_TO_FILE)))
print("Última  :", next(reversed(DATE_TO_FILE)))

if duplicates:
    print("⚠ Datas duplicadas:", len(duplicates))

# =========================================================
# Arquivo de prova
# =========================================================
probe_fp = next(iter(DATE_TO_FILE.values()))

with xr.open_dataset(
    probe_fp,
    engine="netcdf4",
    mask_and_scale=True,
    decode_cf=True
) as ds:

    print("\nVariáveis:", list(ds.data_vars))

    vod_vars = [
        v for v in ds.data_vars
        if "vod" in v.lower()
    ]

    VAR = vod_vars[0] if vod_vars else list(ds.data_vars)[0]

    da = ds[VAR]

    print("\nVariável VOD:", VAR)
    print("dims:", da.dims)
    print("attrs:", da.attrs)

    # nomes das coordenadas
    LAT = "lat" if "lat" in da.coords else "latitude"
    LON = "lon" if "lon" in da.coords else "longitude"

print("\nVAR =", VAR)
print("LAT =", LAT)
print("LON =", LON)

# =========================================================
# Confirmar codificação RAW
# =========================================================
with xr.open_dataset(
    probe_fp,
    engine="netcdf4",
    mask_and_scale=False,
    decode_cf=False
) as ds:

    raw = ds[VAR]

    print("\nMetadados RAW importantes:")

    for key in [
        "_FillValue",
        "missing_value",
        "scale_factor",
        "add_offset",
        "valid_range"
    ]:
        if key in raw.attrs:
            print(f"{key}: {raw.attrs[key]}")

print("\n✔ Esperado para este produto:")
print("  _FillValue = -999999")
print("  faixa válida de VOD = 0–4")

NetCDFs encontrados: 12,497
Datas indexadas: 12,497
Primeira: 1987-07-09 00:00:00
Última  : 2021-12-31 00:00:00

Variáveis: ['VODCA_CXKu']

Variável VOD: VODCA_CXKu
dims: ('time', 'lat', 'lon')
attrs: {'long_name': 'merged VOD from the C-, X-, and Ku band frequencies', 'valid_range': array([0, 4]), 'units': 'unitless'}

VAR = VODCA_CXKu
LAT = lat
LON = lon

Metadados RAW importantes:
_FillValue: -999999.0
valid_range: [0 4]

✔ Esperado para este produto:
  _FillValue = -999999
  faixa válida de VOD = 0–4


In [ ]:
# @title 3 — Funções corrigidas: leitura diária, máscara e composição temporal

# =========================================================
# BBox Pantanal em WGS84
# =========================================================
gdf = gpd.read_file(BOUNDARY_FP).to_crs(4326)

if len(gdf) > 1:
    gdf = gdf.dissolve()

minx, miny, maxx, maxy = gdf.total_bounds

BBOX = (minx, miny, maxx, maxy)

print("BBox Pantanal:")
print(BBOX)

# =========================================================
# Calendário usado no estudo
#
# 01 -> Jan-Apr
# 03 -> Mar-Jun
# 05 -> May-Jun
# 07 -> Jul-Aug
# 09 -> Sep-Oct
# 11 -> Nov-Dec
# =========================================================

ANCHORS = [1, 3, 5, 7, 9, 11]

def period_bounds(yyyymm: str):

    y = int(yyyymm[:4])
    m = int(yyyymm[4:6])

    start = pd.Timestamp(year=y, month=m, day=1)

    n_months = 4 if m in (1, 3) else 2

    end = start + pd.DateOffset(months=n_months)

    return start, end


def files_for_period(yyyymm: str):

    start, end = period_bounds(yyyymm)

    # intervalo [start, end)
    selected = [
        (d, fp)
        for d, fp in DATE_TO_FILE.items()
        if start <= d < end
    ]

    return selected


# =========================================================
# Recorte respeitando orientação das coordenadas
# =========================================================

def slice_bbox(da):

    lon_min, lat_min, lon_max, lat_max = BBOX

    lats = da[LAT].values
    lons = da[LON].values

    # Caso longitude seja 0–360
    if np.nanmin(lons) >= 0 and lon_min < 0:
        lon_min = lon_min % 360
        lon_max = lon_max % 360

    lat_ascending = lats[-1] > lats[0]
    lon_ascending = lons[-1] > lons[0]

    lat_slice = (
        slice(lat_min, lat_max)
        if lat_ascending
        else slice(lat_max, lat_min)
    )

    lon_slice = (
        slice(lon_min, lon_max)
        if lon_ascending
        else slice(lon_max, lon_min)
    )

    return da.sel({
        LAT: lat_slice,
        LON: lon_slice
    })


# =========================================================
# Leitura CORRETA de um arquivo diário
# =========================================================

def read_daily_vod(fp):

    with xr.open_dataset(
        fp,
        engine="netcdf4",

        # CORREÇÃO FUNDAMENTAL
        mask_and_scale=True,
        decode_cf=True

    ) as ds:

        da = ds[VAR]

        if "time" in da.dims:
            da = da.isel(time=0, drop=True)

        da = slice_bbox(da)

        # carrega somente a pequena área Pantanal
        arr = np.asarray(
            da.values,
            dtype="float32"
        )

        lat = np.asarray(
            da[LAT].values,
            dtype="float64"
        )

        lon = np.asarray(
            da[LON].values,
            dtype="float64"
        )

    # =====================================================
    # Guard-rail independente da máscara CF
    # valid_range oficial = 0–4
    # =====================================================
    arr[~np.isfinite(arr)] = np.nan

    arr[(arr < 0) | (arr > 4)] = np.nan

    # longitude crescente
    if lon.size > 1 and lon[0] > lon[-1]:
        lon = lon[::-1]
        arr = arr[:, ::-1]

    # latitude decrescente (N -> S), padrão raster
    if lat.size > 1 and lat[0] < lat[-1]:
        lat = lat[::-1]
        arr = arr[::-1, :]

    return arr, lat, lon


# =========================================================
# Construção de um composto
# =========================================================

def build_composite(yyyymm):

    selected = files_for_period(yyyymm)

    if not selected:
        raise RuntimeError(
            f"Nenhum arquivo encontrado para {yyyymm}"
        )

    sum_arr = None
    count_arr = None

    LAT_REF = None
    LON_REF = None

    good_days = 0
    bad_files = []

    for d, fp in tqdm(
        selected,
        desc=f"VOD {yyyymm}",
        leave=False
    ):

        try:

            arr, lat, lon = read_daily_vod(fp)

            if sum_arr is None:

                sum_arr = np.zeros(
                    arr.shape,
                    dtype="float64"
                )

                count_arr = np.zeros(
                    arr.shape,
                    dtype="uint16"
                )

                LAT_REF = lat
                LON_REF = lon

            else:

                if arr.shape != sum_arr.shape:
                    raise ValueError(
                        f"Shape mudou: {arr.shape} != {sum_arr.shape}"
                    )

            valid = np.isfinite(arr)

            sum_arr[valid] += arr[valid]

            count_arr[valid] += 1

            good_days += 1

        except Exception as e:

            bad_files.append(
                (fp.name, str(e))
            )

    if sum_arr is None:
        raise RuntimeError(
            f"Nenhum dia válido em {yyyymm}"
        )

    mean_arr = np.full(
        sum_arr.shape,
        np.nan,
        dtype="float32"
    )

    ok = count_arr > 0

    mean_arr[ok] = (
        sum_arr[ok] /
        count_arr[ok]
    ).astype("float32")

    # defesa final
    mean_arr[
        (~np.isfinite(mean_arr)) |
        (mean_arr < 0) |
        (mean_arr > 4)
    ] = np.nan

    return {
        "mean": mean_arr,
        "count": count_arr,
        "lat": LAT_REF,
        "lon": LON_REF,
        "n_files": len(selected),
        "good_days": good_days,
        "bad_files": bad_files
    }


# =========================================================
# Escrever GeoTIFF nativo
# =========================================================

def write_native_vod(yyyymm, result):

    arr = result["mean"]
    lat = result["lat"]
    lon = result["lon"]

    dx = float(
        np.nanmedian(np.abs(np.diff(lon)))
    )

    dy = float(
        np.nanmedian(np.abs(np.diff(lat)))
    )

    left = float(lon.min() - dx / 2)
    top  = float(lat.max() + dy / 2)

    transform = from_origin(
        left,
        top,
        dx,
        dy
    )

    out_fp = VOD_BI_DIR / f"vodca_{yyyymm}.tif"

    profile = {
        "driver": "GTiff",
        "height": arr.shape[0],
        "width": arr.shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": "EPSG:4326",
        "transform": transform,

        # SEM sentinel negativo
        "nodata": np.nan,

        "compress": "lzw"
    }

    with rio.open(
        out_fp,
        "w",
        **profile
    ) as dst:

        dst.write(
            arr.astype("float32"),
            1
        )

    return out_fp


# =========================================================
# Estatísticas QC
# =========================================================

def vod_stats(arr):

    v = arr[np.isfinite(arr)]

    if v.size == 0:
        return {
            "n": 0
        }

    return {
        "n": int(v.size),
        "min": float(np.min(v)),
        "p01": float(np.percentile(v, 1)),
        "p50": float(np.percentile(v, 50)),
        "p99": float(np.percentile(v, 99)),
        "max": float(np.max(v))
    }

BBox Pantanal:
(np.float64(-59.14083347699986), np.float64(-22.161549262999934), np.float64(-54.793913081999946), np.float64(-15.646848104999876))


In [ ]:
# @title 4 — PILOTO obrigatório — VOD 201909

PILOT_YM = "201909"

start_time = time.perf_counter()

start, end = period_bounds(PILOT_YM)

print("Período:")
print(start.date(), "→", end.date(), "(fim exclusivo)")

result = build_composite(PILOT_YM)

stats = vod_stats(result["mean"])

elapsed = time.perf_counter() - start_time

print("\n==============================")
print("PILOTO", PILOT_YM)
print("==============================")

print("Arquivos previstos :", result["n_files"])
print("Dias lidos         :", result["good_days"])
print("Arquivos com erro  :", len(result["bad_files"]))

print("\nVOD corrigido:")
for k, v in stats.items():
    print(f"{k:>4}: {v}")

print("\nObservações/dia por pixel:")
cnt = result["count"]
valid_cnt = cnt[cnt > 0]

print("min    :", valid_cnt.min())
print("mediana:", np.median(valid_cnt))
print("max    :", valid_cnt.max())

# Guard-rails
assert stats["n"] > 0, "Nenhum VOD válido."

assert stats["min"] >= 0, (
    f"ERRO: VOD negativo: {stats['min']}"
)

assert stats["max"] <= 4, (
    f"ERRO: VOD > 4: {stats['max']}"
)

out_fp = write_native_vod(
    PILOT_YM,
    result
)

print("\n✔ PILOTO salvo:")
print(out_fp)

print(
    f"\nTempo total: "
    f"{elapsed/60:.2f} min"
)

Período:
2019-09-01 → 2019-11-01 (fim exclusivo)


VOD 201909:   0%|          | 0/61 [00:00<?, ?it/s]


PILOTO 201909
Arquivos previstos : 61
Dias lidos         : 61
Arquivos com erro  : 0

VOD corrigido:
   n: 463
 min: 0.24629521369934082
 p01: 0.3118826746940613
 p50: 0.7256207466125488
 p99: 1.0388776063919067
 max: 1.077331781387329

Observações/dia por pixel:
min    : 20
mediana: 44.0
max    : 48

✔ PILOTO salvo:
/content/drive/MyDrive/Pantanal_TippingPoints/index/interim/vodca_corrected/vodca_201909.tif

Tempo total: 0.78 min


In [ ]:
# @title 5 — PILOTO: alinhamento do VOD corrigido para a grade 300 m

def align_vod_to_reference(
    src_fp,
    out_fp
):

    with rio.open(src_fp) as src, rio.open(REF_FP) as ref:

        src_arr = src.read(1).astype("float32")

        src_arr[
            (~np.isfinite(src_arr)) |
            (src_arr < 0) |
            (src_arr > 4)
        ] = np.nan

        dst = np.full(
            (ref.height, ref.width),
            np.nan,
            dtype="float32"
        )

        reproject(
            source=src_arr,
            destination=dst,

            src_transform=src.transform,
            src_crs=src.crs,

            dst_transform=ref.transform,
            dst_crs=ref.crs,

            src_nodata=np.nan,
            dst_nodata=np.nan,

            # mantém procedimento anterior
            resampling=Resampling.bilinear
        )

        profile = ref.meta.copy()

        profile.update(
            count=1,
            dtype="float32",
            nodata=np.nan,
            compress="lzw",
            tiled=True,
            blockxsize=512,
            blockysize=512,
            BIGTIFF="IF_SAFER"
        )

        with rio.open(
            out_fp,
            "w",
            **profile
        ) as dst_ds:

            dst_ds.write(dst, 1)

    return dst


src_fp = VOD_BI_DIR / f"vodca_{PILOT_YM}.tif"

out_fp = (
    VOD_300_DIR /
    f"vod_{PILOT_YM}_300m_aligned.tif"
)

dst = align_vod_to_reference(
    src_fp,
    out_fp
)

stats = vod_stats(dst)

print("VOD alinhado — QC:")

for k, v in stats.items():
    print(f"{k:>4}: {v}")

print("\nN <= -10000:",
      int(np.sum(
          np.isfinite(dst) &
          (dst <= -10000)
      )))

assert stats["n"] > 0
assert stats["min"] >= 0
assert stats["max"] <= 4

# conferir grade
with rio.open(out_fp) as ds, rio.open(REF_FP) as ref:

    print("\nGrade:")
    print("shape :", ds.shape)
    print("CRS   :", ds.crs)
    print("res   :", ds.res)

    same_shape = ds.shape == ref.shape
    same_crs = str(ds.crs) == str(ref.crs)
    same_transform = np.allclose(
        np.array(ds.transform)[:6],
        np.array(ref.transform)[:6],
        atol=1e-8
    )

    print("\nMesmo shape     :", same_shape)
    print("Mesmo CRS       :", same_crs)
    print("Mesmo transform :", same_transform)

    assert same_shape
    assert same_crs
    assert same_transform

print("\n✔ Piloto 300 m salvo:")
print(out_fp)

VOD alinhado — QC:
   n: 3736578
 min: 0.24629619717597961
 p01: 0.3321586847305298
 p50: 0.7285394668579102
 p99: 1.0219463109970093
 max: 1.077142596244812

N <= -10000: 0

Grade:
shape : (2448, 1612)
CRS   : EPSG:31983
res   : (300.0, 300.0)

Mesmo shape     : True
Mesmo CRS       : True
Mesmo transform : True

✔ Piloto 300 m salvo:
/content/drive/MyDrive/Pantanal_TippingPoints/index/outputs/vod_bi_300m_aligned_corrected/vod_201909_300m_aligned.tif


In [ ]:
# @title QC rápido — disponibilidade VODCA em 1987

dates_1987 = sorted(
    d for d in DATE_TO_FILE
    if d.year == 1987
)

print("Arquivos em 1987:", len(dates_1987))

if dates_1987:
    print("Primeira data:", dates_1987[0].date())
    print("Última data  :", dates_1987[-1].date())

    print("\nArquivos por mês:")
    for m in range(1, 13):
        n = sum(d.month == m for d in dates_1987)
        print(f"{m:02d}: {n}")

Arquivos em 1987: 141
Primeira data: 1987-07-09
Última data  : 1987-11-30

Arquivos por mês:
01: 0
02: 0
03: 0
04: 0
05: 0
06: 0
07: 23
08: 29
09: 30
10: 29
11: 30
12: 0


In [ ]:
 # @title 6 — PROCESSAMENTO COMPLETO VODCA CORRIGIDO 1987–2021 — ROBUSTO

import gc
import time
import re
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio as rio
from tqdm.auto import tqdm


# =========================================================
# PARÂMETROS
# =========================================================

YEARS = list(range(1987, 2022))
ANCHORS = [1, 3, 5, 7, 9, 11]

QC_FP = LOG_DIR / "vodca_corrected_qc.csv"


# =========================================================
# 1. NORMALIZAR ÍNDICE DATA -> ARQUIVO
# =========================================================

DATE_RE = re.compile(r"(19|20)\d{2}-\d{2}-\d{2}")

DATE_TO_FILE_CLEAN = {}

for key, fp in DATE_TO_FILE.items():

    dt = None

    # tenta converter a chave existente
    try:
        dt = pd.Timestamp(key).normalize()
    except Exception:
        pass

    # se necessário, recupera data do nome do arquivo
    if dt is None or pd.isna(dt):

        m = DATE_RE.search(Path(fp).name)

        if m:
            dt = pd.Timestamp(m.group(0)).normalize()

    if dt is not None and not pd.isna(dt):

        DATE_TO_FILE_CLEAN[dt] = Path(fp)


DATE_TO_FILE_CLEAN = dict(
    sorted(
        DATE_TO_FILE_CLEAN.items(),
        key=lambda x: x[0]
    )
)

print(
    "Datas VODCA válidas indexadas:",
    f"{len(DATE_TO_FILE_CLEAN):,}"
)

if DATE_TO_FILE_CLEAN:

    first_date = next(iter(DATE_TO_FILE_CLEAN))
    last_date  = next(reversed(DATE_TO_FILE_CLEAN))

    print("Primeira data:", first_date.date())
    print("Última data  :", last_date.date())


# =========================================================
# 2. VALIDAR UM GeoTIFF JÁ EXISTENTE
#
# Só será considerado checkpoint se:
# - houver pelo menos 1 pixel válido
# - todos os valores válidos estiverem entre 0 e 4
# =========================================================

def existing_vod_is_valid(fp):

    if not fp.exists():
        return False

    try:

        with rio.open(fp) as ds:
            a = ds.read(1).astype("float32")

        v = a[np.isfinite(a)]

        if v.size == 0:
            return False

        if np.min(v) < 0:
            return False

        if np.max(v) > 4:
            return False

        return True

    except Exception:
        return False


# =========================================================
# 3. CARREGAR UM ANO DE VOD DIÁRIO
# =========================================================

def load_year(year):

    entries = [
        (d, fp)
        for d, fp in DATE_TO_FILE_CLEAN.items()
        if d.year == year
    ]

    entries = sorted(
        entries,
        key=lambda x: x[0]
    )

    if not entries:
        return None

    arrays = []
    dates = []

    lat_ref = None
    lon_ref = None
    expected_shape = None

    bad_files = []

    for d, fp in tqdm(
        entries,
        desc=f"Lendo {year}",
        leave=False
    ):

        try:

            arr, lat, lon = read_daily_vod(fp)

            if lat_ref is None:

                lat_ref = lat
                lon_ref = lon
                expected_shape = arr.shape

            else:

                if arr.shape != expected_shape:

                    raise ValueError(
                        f"Shape inconsistente: "
                        f"{arr.shape} != {expected_shape}"
                    )

            arrays.append(
                arr.astype("float32")
            )

            dates.append(
                np.datetime64(d.date())
            )

        except Exception as e:

            bad_files.append(
                (Path(fp).name, str(e))
            )

    if not arrays:
        return None

    stack = np.stack(
        arrays,
        axis=0
    ).astype("float32")

    dates = np.asarray(
        dates,
        dtype="datetime64[D]"
    )

    return {
        "stack": stack,
        "dates": dates,
        "lat": lat_ref,
        "lon": lon_ref,
        "bad": bad_files,
        "n_files": len(entries)
    }


# =========================================================
# 4. GERAR UM COMPOSTO TEMPORAL
# =========================================================

def composite_from_year(year_data, yyyymm):

    if year_data is None:
        return None, None, 0

    start, end = period_bounds(yyyymm)

    start64 = np.datetime64(
        start.date()
    )

    end64 = np.datetime64(
        end.date()
    )

    dates = year_data["dates"]

    select = (
        (dates >= start64) &
        (dates < end64)
    )

    n_days = int(
        np.sum(select)
    )

    # -----------------------------------------
    # nenhum arquivo diário nesse período
    # -----------------------------------------
    if n_days == 0:
        return None, None, 0

    sub = year_data["stack"][select]

    # -----------------------------------------
    # somente valores realmente válidos
    # NaN NÃO entra nem na soma nem no divisor
    # -----------------------------------------
    valid = np.isfinite(sub)

    count = valid.sum(
        axis=0
    ).astype("uint16")

    sums = np.nansum(
        sub,
        axis=0,
        dtype="float64"
    )

    mean = np.full(
        count.shape,
        np.nan,
        dtype="float32"
    )

    ok = count > 0

    mean[ok] = (
        sums[ok] /
        count[ok]
    ).astype("float32")

    # -----------------------------------------
    # HARD QC FINAL
    # faixa física oficial VODCA = 0–4
    # -----------------------------------------
    mean[
        (~np.isfinite(mean)) |
        (mean < 0) |
        (mean > 4)
    ] = np.nan

    return mean, count, n_days


# =========================================================
# 5. CARREGAR QC EXISTENTE
# =========================================================

if QC_FP.exists():

    try:

        qc_table = pd.read_csv(
            QC_FP,
            dtype={"yyyymm": "string"}
        )

        if "yyyymm" in qc_table.columns:

            qc_table["yyyymm"] = (
                qc_table["yyyymm"]
                .astype(str)
                .str.replace(
                    r"\.0$",
                    "",
                    regex=True
                )
                .str.zfill(6)
            )

    except Exception as e:

        print(
            "⚠ Não foi possível ler QC anterior:",
            e
        )

        qc_table = pd.DataFrame()

else:

    qc_table = pd.DataFrame()


# converte tabela existente em lista de registros
if not qc_table.empty:
    qc_records = qc_table.to_dict("records")
else:
    qc_records = []


# =========================================================
# 6. FUNÇÃO PARA ATUALIZAR O QC
# =========================================================

def save_qc_record(record):

    global qc_records

    ym = str(record["yyyymm"]).zfill(6)

    record["yyyymm"] = ym

    # remove registro antigo desse mesmo período
    qc_records = [
        r for r in qc_records
        if str(r.get("yyyymm", "")).zfill(6) != ym
    ]

    qc_records.append(record)

    df = pd.DataFrame(qc_records)

    if "yyyymm" in df.columns:

        df["yyyymm"] = (
            df["yyyymm"]
            .astype(str)
            .str.replace(
                r"\.0$",
                "",
                regex=True
            )
            .str.zfill(6)
        )

        df["_sort"] = pd.to_numeric(
            df["yyyymm"],
            errors="coerce"
        )

        df = (
            df
            .sort_values("_sort")
            .drop(columns="_sort")
        )

    df.to_csv(
        QC_FP,
        index=False
    )


# =========================================================
# 7. LOOP PRINCIPAL
# =========================================================

for year in tqdm(
    YEARS,
    desc="VODCA corrigido por ano"
):

    target_yms = [
        f"{year}{m:02d}"
        for m in ANCHORS
    ]

    t_year = time.perf_counter()

    # -----------------------------------------------------
    # Primeiro verificar se todos os arquivos existentes
    # desse ano são realmente válidos
    # -----------------------------------------------------

    periods_to_process = []
    already_exists = 0

    for ym in target_yms:

        out_fp = (
            VOD_BI_DIR /
            f"vodca_{ym}.tif"
        )

        if out_fp.exists():

            if existing_vod_is_valid(out_fp):

                already_exists += 1
                continue

            else:

                print(
                    f"  ⚠ {ym}: "
                    "arquivo existente inválido/vazio — será removido."
                )

                try:
                    out_fp.unlink()
                except Exception:
                    pass

        periods_to_process.append(ym)


    # -----------------------------------------------------
    # Se todos os produtos existentes são válidos,
    # não precisa sequer carregar o ano
    # -----------------------------------------------------

    if len(periods_to_process) == 0:

        print(
            f"✔ {year}: "
            f"todos os {already_exists} períodos disponíveis "
            f"já estavam válidos."
        )

        continue


    # -----------------------------------------------------
    # carregar VOD diário do ano
    # -----------------------------------------------------

    yd = load_year(year)

    if yd is None:

        print(
            f"\n⚠ {year}: "
            "nenhum arquivo VODCA diário disponível."
        )

        continue


    print(
        f"\n{year}: "
        f"{yd['n_files']} arquivos | "
        f"falhas={len(yd['bad'])}"
    )


    generated_this_year = 0
    skipped_no_files = 0
    skipped_no_valid = 0


    # =====================================================
    # PROCESSAR CADA ÂNCORA
    # =====================================================

    for ym in periods_to_process:

        out_fp = (
            VOD_BI_DIR /
            f"vodca_{ym}.tif"
        )


        mean, count, n_days = composite_from_year(
            yd,
            ym
        )


        # -------------------------------------------------
        # CASO A:
        # não existem dias naquele período
        # -------------------------------------------------

        if mean is None:

            skipped_no_files += 1

            print(
                f"  ⚠ {ym}: "
                "sem arquivos VODCA no período — ignorado."
            )

            save_qc_record({
                "yyyymm": ym,
                "status": "no_daily_files",
                "n_days": 0,
                "valid_pixels": 0,
                "vod_min": np.nan,
                "vod_p01": np.nan,
                "vod_p50": np.nan,
                "vod_p99": np.nan,
                "vod_max": np.nan,
                "nobs_min": 0,
                "nobs_median": np.nan,
                "nobs_max": 0
            })

            continue


        # -------------------------------------------------
        # QC ANTES DE SALVAR
        # -------------------------------------------------

        stats = vod_stats(mean)


        # -------------------------------------------------
        # CASO B:
        # há arquivos diários,
        # mas nenhum VOD válido no Pantanal
        # -------------------------------------------------

        if stats["n"] == 0:

            skipped_no_valid += 1

            print(
                f"  ⚠ {ym}: "
                f"{n_days} dias disponíveis, "
                "mas nenhum VOD válido (0–4) no Pantanal — ignorado."
            )

            # garante que nenhum raster vazio permaneça
            try:
                if out_fp.exists():
                    out_fp.unlink()
            except Exception:
                pass

            save_qc_record({
                "yyyymm": ym,
                "status": "no_valid_vod",
                "n_days": n_days,
                "valid_pixels": 0,
                "vod_min": np.nan,
                "vod_p01": np.nan,
                "vod_p50": np.nan,
                "vod_p99": np.nan,
                "vod_max": np.nan,
                "nobs_min": 0,
                "nobs_median": np.nan,
                "nobs_max": 0
            })

            continue


        # -------------------------------------------------
        # HARD QC
        # estes casos NÃO devem ser simplesmente ignorados
        # -------------------------------------------------

        if stats["min"] < 0:

            raise RuntimeError(
                f"QC falhou em {ym}: "
                f"VOD mínimo = {stats['min']}"
            )


        if stats["max"] > 4:

            raise RuntimeError(
                f"QC falhou em {ym}: "
                f"VOD máximo = {stats['max']}"
            )


        # -------------------------------------------------
        # SOMENTE AGORA salvar GeoTIFF válido
        # -------------------------------------------------

        result = {
            "mean": mean,
            "count": count,
            "lat": yd["lat"],
            "lon": yd["lon"]
        }


        write_native_vod(
            ym,
            result
        )


        # -------------------------------------------------
        # VALIDAÇÃO DO ARQUIVO GRAVADO
        # -------------------------------------------------

        if not existing_vod_is_valid(out_fp):

            try:
                if out_fp.exists():
                    out_fp.unlink()
            except Exception:
                pass

            raise RuntimeError(
                f"O GeoTIFF {ym} foi gravado, "
                "mas falhou no QC pós-escrita."
            )


        valid_count = count[count > 0]


        save_qc_record({
            "yyyymm": ym,
            "status": "valid",
            "n_days": n_days,
            "valid_pixels": stats["n"],
            "vod_min": stats["min"],
            "vod_p01": stats["p01"],
            "vod_p50": stats["p50"],
            "vod_p99": stats["p99"],
            "vod_max": stats["max"],
            "nobs_min": (
                int(valid_count.min())
                if valid_count.size
                else 0
            ),
            "nobs_median": (
                float(np.median(valid_count))
                if valid_count.size
                else np.nan
            ),
            "nobs_max": (
                int(valid_count.max())
                if valid_count.size
                else 0
            )
        })


        generated_this_year += 1


    # =====================================================
    # RESUMO ANUAL
    # =====================================================

    elapsed = (
        time.perf_counter() -
        t_year
    ) / 60


    print(
        f"✔ {year}: "
        f"novos={generated_this_year} | "
        f"já válidos={already_exists} | "
        f"sem arquivos={skipped_no_files} | "
        f"sem VOD válido={skipped_no_valid} | "
        f"tempo={elapsed:.1f} min"
    )


    del yd
    gc.collect()


# =========================================================
# 8. CONFERÊNCIA FINAL
# =========================================================

outputs = sorted(
    VOD_BI_DIR.glob(
        "vodca_??????.tif"
    )
)


valid_outputs = []

invalid_outputs = []


for fp in outputs:

    if existing_vod_is_valid(fp):

        valid_outputs.append(fp)

    else:

        invalid_outputs.append(fp)


print("\n" + "="*65)
print("PROCESSAMENTO FINALIZADO")
print("="*65)


print(
    "GeoTIFFs válidos:",
    len(valid_outputs)
)

print(
    "GeoTIFFs inválidos:",
    len(invalid_outputs)
)


if valid_outputs:

    periods = [
        re.search(
            r"\d{6}",
            p.name
        ).group(0)
        for p in valid_outputs
    ]

    print(
        "Primeiro período válido:",
        periods[0]
    )

    print(
        "Último período válido :",
        periods[-1]
    )


if invalid_outputs:

    print(
        "\n⚠ Arquivos inválidos encontrados:"
    )

    for fp in invalid_outputs[:20]:
        print(" ", fp.name)


print(
    "\nQC salvo em:",
    QC_FP
)


print("\n✔ _FillValue não participa da média.")
print("✔ NaN não participa da soma nem do denominador.")
print("✔ Períodos sem VOD válido permanecem ausentes.")
print("✔ Somente VOD entre 0 e 4 é aceito.")
print("✔ Arquivos existentes são validados antes de serem reutilizados.")

Datas VODCA válidas indexadas: 12,497
Primeira data: 1987-07-09
Última data  : 2021-12-31


VODCA corrigido por ano:   0%|          | 0/35 [00:00<?, ?it/s]

Lendo 1987:   0%|          | 0/141 [00:00<?, ?it/s]


1987: 141 arquivos | falhas=0
  ⚠ 198701: sem arquivos VODCA no período — ignorado.
  ⚠ 198703: sem arquivos VODCA no período — ignorado.
  ⚠ 198705: sem arquivos VODCA no período — ignorado.
✔ 1987: novos=0 | já válidos=3 | sem arquivos=3 | sem VOD válido=0 | tempo=2.0 min
✔ 1988: todos os 6 períodos disponíveis já estavam válidos.
✔ 1989: todos os 6 períodos disponíveis já estavam válidos.
✔ 1990: todos os 6 períodos disponíveis já estavam válidos.


Lendo 1991:   0%|          | 0/359 [00:00<?, ?it/s]

...


2019: 365 arquivos | falhas=0
✔ 2019: novos=5 | já válidos=1 | sem arquivos=0 | sem VOD válido=0 | tempo=4.8 min


Lendo 2020:   0%|          | 0/366 [00:00<?, ?it/s]


2020: 366 arquivos | falhas=0
✔ 2020: novos=6 | já válidos=0 | sem arquivos=0 | sem VOD válido=0 | tempo=4.9 min


Lendo 2021:   0%|          | 0/365 [00:00<?, ?it/s]


2021: 365 arquivos | falhas=0
✔ 2021: novos=6 | já válidos=0 | sem arquivos=0 | sem VOD válido=0 | tempo=5.0 min

PROCESSAMENTO FINALIZADO
GeoTIFFs válidos: 206
GeoTIFFs inválidos: 0
Primeiro período válido: 198707
Último período válido : 202111

QC salvo em: /content/drive/MyDrive/Pantanal_TippingPoints/index/logs/vodca_corrected/vodca_corrected_qc.csv

✔ _FillValue não participa da média.
✔ NaN não participa da soma nem do denominador.
✔ Períodos sem VOD válido permanecem ausentes.
✔ Somente VOD entre 0 e 4 é aceito.
✔ Arquivos existentes são validados antes de serem reutilizados.


In [ ]:
# @title 7 — QC GLOBAL dos VOD temporais corrigidos

files = sorted(
    VOD_BI_DIR.glob(
        "vodca_??????.tif"
    )
)

rows = []
bad = []

for fp in tqdm(
    files,
    desc="QC VOD corrigido"
):

    with rio.open(fp) as ds:
        a = ds.read(1).astype("float32")

    v = a[np.isfinite(a)]

    if v.size == 0:
        bad.append(
            (fp.name, "sem dados")
        )
        continue

    mn = float(v.min())
    mx = float(v.max())
    med = float(np.median(v))

    if mn < 0 or mx > 4:
        bad.append(
            (fp.name, f"{mn}..{mx}")
        )

    rows.append({
        "file": fp.name,
        "n": v.size,
        "min": mn,
        "median": med,
        "max": mx
    })

qa = pd.DataFrame(rows)

print("Arquivos verificados:", len(files))
print("Problemas:", len(bad))

if bad:
    print("\nPRIMEIROS PROBLEMAS:")
    for x in bad[:20]:
        print(x)

print("\nResumo das medianas:")
print(
    qa["median"].describe()
)

assert len(bad) == 0, (
    "Há arquivos fora da faixa 0–4. "
    "NÃO prossiga para alinhamento."
)

print("\n✔ Todos os VOD estão dentro de 0–4.")

QC VOD corrigido:   0%|          | 0/206 [00:00<?, ?it/s]

Arquivos verificados: 206
Problemas: 0

Resumo das medianas:
count    206.000000
mean       0.756387
std        0.036836
min        0.659375
25%        0.732244
50%        0.758530
75%        0.783681
max        0.842299
Name: median, dtype: float64

✔ Todos os VOD estão dentro de 0–4.


In [ ]:
# @title 8 — Alinhar VOD corrigido à grade 300 m

native_files = sorted(
    VOD_BI_DIR.glob(
        "vodca_??????.tif"
    )
)

ALIGN_QC_FP = (
    LOG_DIR /
    "vodca_300m_corrected_qc.csv"
)

align_rows = []

for src_fp in tqdm(
    native_files,
    desc="VOD → grade 300 m"
):

    ym = re.search(
        r"\d{6}",
        src_fp.name
    ).group(0)

    out_fp = (
        VOD_300_DIR /
        f"vod_{ym}_300m_aligned.tif"
    )

    if out_fp.exists():
        continue

    dst = align_vod_to_reference(
        src_fp,
        out_fp
    )

    stats = vod_stats(dst)

    if (
        stats["n"] == 0 or
        stats["min"] < -1e-6 or
        stats["max"] > 4 + 1e-6
    ):
        raise RuntimeError(
            f"QC alinhado falhou em "
            f"{ym}: {stats}"
        )

    align_rows.append({
        "yyyymm": ym,
        "valid_pixels": stats["n"],
        "min": stats["min"],
        "p01": stats["p01"],
        "p50": stats["p50"],
        "p99": stats["p99"],
        "max": stats["max"]
    })

    if len(align_rows) % 10 == 0:

        pd.DataFrame(
            align_rows
        ).to_csv(
            ALIGN_QC_FP,
            index=False
        )

# salvar final
if align_rows:

    pd.DataFrame(
        align_rows
    ).to_csv(
        ALIGN_QC_FP,
        index=False
    )

files300 = sorted(
    VOD_300_DIR.glob(
        "vod_??????_300m_aligned.tif"
    )
)

print("\nArquivos alinhados:", len(files300))
print("Pasta:", VOD_300_DIR)
print("QC:", ALIGN_QC_FP)

VOD → grade 300 m:   0%|          | 0/206 [00:00<?, ?it/s]


Arquivos alinhados: 206
Pasta: /content/drive/MyDrive/Pantanal_TippingPoints/index/outputs/vod_bi_300m_aligned_corrected
QC: /content/drive/MyDrive/Pantanal_TippingPoints/index/logs/vodca_corrected/vodca_300m_corrected_qc.csv


In [ ]:
# @title 9 — QC FINAL do VOD corrigido 300 m

files = sorted(
    VOD_300_DIR.glob(
        "vod_??????_300m_aligned.tif"
    )
)

with rio.open(REF_FP) as ref:
    REF_SHAPE = ref.shape
    REF_CRS = str(ref.crs)
    REF_TRANSFORM = np.array(
        ref.transform
    )[:6]

problems = []

for fp in tqdm(
    files,
    desc="QC final"
):

    with rio.open(fp) as ds:

        if ds.shape != REF_SHAPE:
            problems.append(
                (fp.name, "shape")
            )
            continue

        if str(ds.crs) != REF_CRS:
            problems.append(
                (fp.name, "CRS")
            )
            continue

        if not np.allclose(
            np.array(ds.transform)[:6],
            REF_TRANSFORM,
            atol=1e-8
        ):
            problems.append(
                (fp.name, "transform")
            )
            continue

        a = ds.read(1).astype(
            "float32"
        )

        v = a[np.isfinite(a)]

        if not v.size:
            problems.append(
                (fp.name, "sem dados")
            )
            continue

        if (
            np.min(v) < -1e-6 or
            np.max(v) > 4 + 1e-6
        ):
            problems.append(
                (
                    fp.name,
                    f"range={v.min()}..{v.max()}"
                )
            )

print("\n================================")
print("QC FINAL")
print("================================")

print("Arquivos:", len(files))
print("Problemas:", len(problems))

if problems:
    for p in problems[:20]:
        print(p)

assert len(problems) == 0

print("\n✔ VOD corrigido validado.")
print("✔ mesma grade 300 m do estudo.")
print("✔ sem -999999.")
print("✔ sem valores negativos absurdos.")
print("✔ faixa física preservada: 0–4.")

QC final:   0%|          | 0/206 [00:00<?, ?it/s]


QC FINAL
Arquivos: 206
Problemas: 0

✔ VOD corrigido validado.
✔ mesma grade 300 m do estudo.
✔ sem -999999.
✔ sem valores negativos absurdos.
✔ faixa física preservada: 0–4.
